# Day 4.2: MLOps Model Serving and Prediction Dashboard

This notebook connects the previous days into one local workflow.

The key question is:

> We have collected data, served data through APIs, scheduled refresh jobs, trained models, and tracked models in MLflow. How do these pieces become one dashboard?

This notebook has two parts:

1. Review the data product: scheduled data refresh + stable data serving + dashboard.
2. Add the model product: MLflow Model Registry + model loading + prediction + prediction logging.


## Big Picture

We are not trying to make Day 1 complicated retroactively. Day 1 was intentionally simple.

Now that we are on Day 4, we can name the layers more clearly:

```text
Data collection schedule
-> Local database
-> Data serving API
-> Dashboard
-> Feature creation
-> MLflow registered model
-> Prediction API
-> Prediction logs
```

In a production system, these may be separate services. For this local demo, we keep them small and runnable.


## Review From Previous Days

**Day 1** gave us a first dashboard:

```text
API data -> local database -> FastAPI -> dashboard
```

**Day 2** introduced two important engineering ideas:

- **Data serving**: stable API endpoints that the dashboard can call.
- **Scheduling**: repeated jobs that refresh data without manual notebook clicks.

Airflow is one common scheduler. For this Day 4 local demo, we use a lightweight scheduler script so everyone can run it quickly. The concept is the same:

```text
Every N minutes, refresh the data source.
```


## Part 1 - Refresh Dashboard Data In The Same Docker Container

For model serving, the dashboard needs recent taxi/weather records to build one prediction request.

In the unified Day 4 setup, the dashboard database you can inspect is stored in:

```text
day_4/day_4_mlflow/live_dashboard_data.db
```

Inside Docker, the scheduler writes to `/tmp/live_dashboard_data.db` first, then syncs the result back to `/workspace/live_dashboard_data.db`. This avoids SQLite disk I/O issues on Windows/OneDrive bind mounts.

The refresh script is also inside the same Docker working folder:

```text
day_4/day_4_mlflow/live_data_scheduler.py
```

This means the whole Day 4 flow can run inside one container:

```text
MLflow server
training scripts
data refresh script
dashboard API
feature-store demo
```

Make sure the unified container is running first:

```powershell
cd day_4\day_4_mlflow
docker compose up --build -d
```




## Run One Data Refresh First

Run one refresh inside the existing `day4-mlops` container:



In [ ]:
%pwd

In [41]:
!docker exec -w /workspace day4-mlops python live_data_scheduler.py --run-once --source all --verbose


[taxi] inserted=52; taxi points=3022; planning-area rows=52
[weather] inserted=47; forecast rows=47
[realtime_weather] inserted=77; rainfall rows=77
Synced runtime database back to: /workspace/live_dashboard_data.db


Expected result:

- taxi rows are inserted
- 2-hour forecast rows are inserted
- realtime weather rows are inserted
- `/tmp/live_dashboard_data.db` is updated inside the container
- `day_4/day_4_mlflow/live_dashboard_data.db` is synced back for inspection

This is the local data source for the dashboard.



## Optional: Keep Refreshing Data

To keep the data updating, open a terminal and enter the same container:

```powershell
docker exec -it -w /workspace day4-mlops bash
```

Inside the container, run:

```bash
python live_data_scheduler.py
```

Leave that terminal open. You will see the refresh logs directly. The script keeps writing to `/tmp/live_dashboard_data.db` and syncing back to `/workspace/live_dashboard_data.db`.



The full scheduler repeats the refresh schedule:

```text
taxi + realtime weather: every 2 minutes
2-hour weather forecast: every 30 minutes
```

In a real production workflow, this role could be handled by Airflow, cron, or another orchestrator. Here we keep it lightweight because the main Day 4 topic is MLOps.



## Part 2 - Start The Data Serving And Dashboard API

The MLflow server is already running inside the unified container:

```text
day4-mlops
```

Open a terminal and enter the same container:

```powershell
docker exec -it -w /workspace day4-mlops bash
```

Inside the container, start the dashboard API:

```bash
cp /workspace/live_dashboard_data.db /tmp/live_dashboard_data.db
DASHBOARD_DB_PATH=/tmp/live_dashboard_data.db \
MLFLOW_TRACKING_URI=http://127.0.0.1:5000 \
MLFLOW_MODEL_URI=models:/taxi_availability_2h_model@champion \
python -m uvicorn dashboard_app:app --host 0.0.0.0 --port 8000
```

Keep this terminal open while using the dashboard.

Open:

```text
http://localhost:8000/predict-dashboard
```

Why this is simpler:

```text
MLflow server, training scripts, dashboard app, mlflow.db, and artifacts all see the same folder: /workspace.
```

So the model artifact path recorded by MLflow is also readable by the dashboard.

Why copy the database to `/tmp` first?

```bash
cp /workspace/live_dashboard_data.db /tmp/live_dashboard_data.db
```

`/workspace/live_dashboard_data.db` is the database file in the shared project folder. On Windows, this folder is a Docker bind mount, and SQLite can sometimes be slow or fragile when a container reads a large database directly from a mounted Windows/OneDrive path.

`/tmp/live_dashboard_data.db` is a copy inside the container's own Linux filesystem. The dashboard only needs a stable local snapshot for reading, so using `/tmp` makes the demo more reliable.

That is why we set:

```bash
DASHBOARD_DB_PATH=/tmp/live_dashboard_data.db
```

The serving app reads taxi/weather rows from this copied database, builds online features, loads the MLflow champion model, and returns predictions.

Every 2 minutes, the dashboard page automatically calls `/api/data/refresh`, runs one live data collection, and redraws the charts. The **Refresh data** button does the same thing manually.




If the dashboard terminal is already running and you refreshed `live_dashboard_data.db`, stop the dashboard with `Ctrl+C`, then run the same two commands again:

```bash
cp /workspace/live_dashboard_data.db /tmp/live_dashboard_data.db
DASHBOARD_DB_PATH=/tmp/live_dashboard_data.db MLFLOW_TRACKING_URI=http://127.0.0.1:5000 MLFLOW_MODEL_URI=models:/taxi_availability_2h_model@champion python -m uvicorn dashboard_app:app --host 0.0.0.0 --port 8000
```



## Updating Files While Docker Is Running

Most Day 4 files are mounted into the container:

```text
local day_4/day_4_mlflow  <->  container /workspace
```

So if you edit a Python file, CSV file, notebook, or SQLite database in this folder, you usually do **not** need `docker compose down`.

Use the lightest reset that matches what changed:

| What changed? | What to do |
|---|---|
| Training script or CSV file | Run the training command again. |
| `dashboard_app.py` | Stop the dashboard API with `Ctrl+C`, then start `uvicorn` again. |
| `live_dashboard_data.db` | Re-copy it into `/tmp`, then restart the dashboard API. |
| Champion alias changed in MLflow | Use the dashboard's reload endpoint/button, or restart the dashboard API. |
| `Dockerfile`, `requirements`, or installed packages | Rebuild with `docker compose up --build -d`. |
| You want to stop everything | Run `docker compose down`. |




## Open the Review Dashboard

Open:

```text
http://localhost:8000/review
```

This page reviews the data product:

- current taxi count
- current weather forecast
- historical taxi curve
- stable API endpoints behind the dashboard

At this point, it is still a data dashboard. It does not make a model prediction yet.


## Data Serving Endpoints

The dashboard calls API endpoints like these:

```text
/api/areas
/api/taxi/current
/api/weather/current
/api/taxi/timeseries
/api/freshness
```

Key idea:

> The browser does not read the database directly. It calls a stable API. This is data serving.


In [ ]:
import requests


def show_response(response, max_chars=1200):
    print("status_code:", response.status_code)
    print("content_type:", response.headers.get("content-type"))
    if response.text:
        print("text_preview:", response.text[:max_chars])
    else:
        print("text_preview: <empty response body>")

    try:
        return response.json()
    except ValueError:
        print("Response is not JSON. Check the status code and text preview above.")
        return None


response = requests.get("http://localhost:8000/api/freshness", timeout=20)
show_response(response)


## What Is Still Missing?

The dashboard can show what is happening now.

But an operational dashboard often needs a forward-looking question:

> How many taxis might be available two hours later?

To answer that, we need model serving.


## Part 3 - Prepare a Registered MLflow Model

This notebook uses the same `day4-mlops` container from Day 4.1.

The prediction dashboard loads this model URI:

```text
models:/taxi_availability_2h_model@champion
```

Meaning:

- `taxi_availability_2h_model` is the registered model name.
- `champion` is the alias for the model version we want the app to use.

First check that the container can see the unified working folder:



In [ ]:
%pwd

In [ ]:
!docker exec day4-mlops ls /workspace


__pycache__
artifacts
day_4_mlflow
mlflow.db
prediction_logs.db
prepare_2h_prediction_dataset.py
serve_model.py
taxi_weather_2h_prediction_dataset.csv
taxi_weather_2h_prediction_dataset_0606.csv
taxi_weather_prediction_dataset.csv
train_linear_regression.py
train_linear_regression_mlflow.py
train_mlp_regression_mlflow.py


You should see files such as:

```text
train_linear_regression_mlflow.py
train_mlp_regression_mlflow.py
taxi_weather_2h_prediction_dataset_0606.csv
dashboard_app.py
feature_store_demo
```

If the model is not registered yet, run this command in the same Docker container:



In [ ]:
%pwd

In [23]:
!docker exec -w /workspace day4-mlops python train_linear_regression_mlflow.py --data-version v1 --feature-version v1 --register-model --model-name taxi_availability_2h_model --model-alias champion


Taxi Availability Prediction: MLflow Training Script
MLflow Tracking URI: http://localhost:5000
Experiment name: taxi_availability_2h_prediction
Run name: linear_regression_data_v1_features_v1
Data version: v1
Dataset path: /mlflow/taxi_weather_2h_prediction_dataset_0606.csv
Rows: 65374
Feature version: v1
Features: hour,day_of_week,is_weekend,is_rain_forecast_2h,weather_category_id,current_taxi_count
Target column: target_taxi_count_2h

Training Linear Regression
------------------------------------------------------------------------

Model Registry
------------------------------------------------------------------------
Registered model name: taxi_availability_2h_model
Registered model version: 4
Alias: champion
Model URI: models:/taxi_availability_2h_model@champion

Final test results
------------------------------------------------------------------------
MAE:  21.426
RMSE: 38.143
R2:   0.736

Logged to MLflow. Open http://localhost:5000 and inspect the run.
🏃 View run linear_regr

/usr/local/lib/python3.10/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.10/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

Open MLflow UI:

```text
http://localhost:5000
```

Check:

- the run exists
- parameters and metrics are logged
- the model is registered
- alias `champion` points to a model version


## Part 4 - Open the Prediction Dashboard

Open:

```text
http://localhost:8000/predict-dashboard
```

Click **Predict 2h taxi count**.

The page now demonstrates a model product:

```text
Dashboard
-> FastAPI prediction endpoint
-> latest taxi/weather data
-> feature creation
-> MLflow champion model
-> prediction result
-> prediction log
```


## Online Prediction Path: Load Model, Build Features, Predict

This is the most important production idea in this notebook.

When the dashboard calls:

```text
/api/model/predict-2h?area=ANG%20MO%20KIO
```

the API is **not training a model again**. It is doing online inference:

```text
request area
-> read latest taxi/weather rows
-> build the same feature columns used in training
-> load or reuse the MLflow champion model
-> call model.predict(model_input)
-> return JSON response
-> write a prediction log
```

In the code, look at:

| What to look for | File / function | What it does |
|---|---|---|
| Model pointer | `day_4_mlflow/dashboard_app.py`, `MLFLOW_MODEL_URI` | Points to `models:/taxi_availability_2h_model@champion`. |
| Model loading | `load_mlflow_model()` | Resolves the champion model and caches it so every request does not reload from MLflow. |
| Feature generation | `get_latest_feature_row(area)` | Turns latest taxi/weather data into model input columns. |
| Prediction endpoint | `/api/model/predict-2h` | Connects data lookup, feature generation, model loading, prediction, response, and logging. |
| Actual inference | `model.predict(model_input)` | Runs the loaded model on the online feature row. |
| Prediction logging | `append_prediction_log(...)` | Saves what happened after deployment for monitoring/debugging. |

The key contract is: **the online feature columns must match the feature columns used during training**.

For this Day 4.2 demo, the serving API builds features manually. In Day 4.3, we introduce Feast to make this feature contract more explicit and reusable.






In [ ]:
response = requests.get(
    "http://localhost:8000/api/model/predict-2h",
    params={"area": "ANG MO KIO"},
    timeout=60,
)
show_response(response)


## Prediction Logging

Prediction logs are written to an append-only JSON Lines file:

```text
day_4/day_4_mlflow/prediction_logs.jsonl
```

Each line is one prediction event. This is a simple monitoring starting point. We save:

- timestamp
- model URI, model version, and model type
- feature version
- input features
- raw and rounded prediction output
- model load time and inference time

This is different from MLflow experiment tracking. MLflow tracks training runs; this file tracks model usage after the model is served.



In [ ]:
import pandas as pd
from pathlib import Path

log_path = Path("day_4_mlflow/prediction_logs.jsonl")
if log_path.exists():
    display(pd.read_json(log_path, lines=True).tail(10))
else:
    print("No prediction log yet. Click the prediction button first.")


## Discussion Questions

Questions:

- Which part is data collection?
- Which part is data serving?
- Which part is model serving?
- Where is the model loaded from?
- Where are online features created?
- Why must online feature columns match training feature columns?
- Why do we need MLflow Model Registry?
- What does the `champion` alias buy us?
- What should we log after a prediction?

This is the Day 4 connection back to Day 1 and Day 2.




## Part 5 - Demo: Update the Served Model

Now we can demonstrate the reason for using a registered model name and an alias.

The dashboard does **not** need to know whether the model is Linear Regression or MLP. It only loads:

```text
models:/taxi_availability_2h_model@champion
```

For this demo:

1. Check which version `champion` points to now.
2. Train an MLP model.
3. Register the MLP under the same registered model name: `taxi_availability_2h_model`.
4. Move the `champion` alias to the new MLP version.
5. Reload the FastAPI model cache.
6. Call the prediction dashboard again.

This is model updating without changing the dashboard code.



### 5.1 Check the Current Champion

Run this in the notebook after MLflow is running. It tells us which model version is currently selected by the alias.



In [ ]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
client = mlflow.tracking.MlflowClient()

current = client.get_model_version_by_alias(
    "taxi_availability_2h_model",
    "champion",
)
run = client.get_run(current.run_id)

{
    "model_name": current.name,
    "champion_version": current.version,
    "run_id": current.run_id,
    "run_name": run.data.tags.get("mlflow.runName"),
    "model_type": run.data.params.get("model_type"),
    "MAE": run.data.metrics.get("MAE"),
    "RMSE": run.data.metrics.get("RMSE"),
    "R2": run.data.metrics.get("R2"),
}


### 5.2 Train and Register the MLP as the New Champion

The important part is not the algorithm name. The important part is the registered model name:

```text
taxi_availability_2h_model
```

Both Linear Regression and MLP are versions of the same business model: predicting 2-hour taxi availability.

Run this in a terminal from the project root. The command registers the MLP model and moves `champion` to the new version.



In [ ]:
%pwd

In [ ]:
!docker exec -w /workspace day4-mlops python train_mlp_regression_mlflow.py --data-version v1 --feature-version v1 --register-model --model-name taxi_availability_2h_model --model-alias champion


### 5.3 Confirm the Alias Moved

Run the same registry check again. The `model_type` should now be `MLPRegressor`, and the champion version should be newer.



In [ ]:
updated = client.get_model_version_by_alias(
    "taxi_availability_2h_model",
    "champion",
)
updated_run = client.get_run(updated.run_id)

{
    "model_name": updated.name,
    "champion_version": updated.version,
    "run_id": updated.run_id,
    "run_name": updated_run.data.tags.get("mlflow.runName"),
    "model_type": updated_run.data.params.get("model_type"),
    "MAE": updated_run.data.metrics.get("MAE"),
    "RMSE": updated_run.data.metrics.get("RMSE"),
    "R2": updated_run.data.metrics.get("R2"),
}


### 5.4 Reload the Serving App Model Cache

The FastAPI app caches the loaded model so predictions are fast. After moving the alias, tell the app to reload the champion model.

For this local demo, click **Reload champion model** on the prediction dashboard:

```text
http://localhost:8000/predict-dashboard
```

The button calls the same API endpoint below. This is the serving-side update step.



In [ ]:
response = requests.post("http://localhost:8000/api/model/reload", timeout=60)
show_response(response)


### 5.5 Verify What the Dashboard Is Serving

This endpoint shows the registered model name, alias, version, run name, and model type currently selected by the serving app.



In [ ]:
response = requests.get("http://localhost:8000/api/model/info", timeout=30)
show_response(response)


### 5.6 Call the Prediction Endpoint Again

The dashboard URL did not change. The API URL did not change. Only the model version behind `champion` changed.



In [ ]:
response = requests.get(
    "http://localhost:8000/api/model/predict-2h",
    params={"area": "ANG MO KIO"},
    timeout=60,
)
show_response(response)


### Key Idea

This is the core model-serving idea:

```text
Application code stays stable.
Model Registry changes which trained model is served.
Alias decides the active version.
Serving layer reloads the selected version.
```

That is why Linear Regression and MLP should register under the same model name when they serve the same prediction task.



## Summary

```text
Scheduled data refresh
-> local database
-> data serving API
-> dashboard
-> MLflow experiment tracking
-> model registry
-> champion alias
-> model serving endpoint
-> prediction dashboard
-> prediction logging
```

This is the MLOps dashboard story: not just a chart, but a small working system that connects data, models, and application use.
